In [1]:
import pandas as pd
import glob
import os

In [2]:
#Create a list of all the CSV files in the raw data folder
path = r'Data/01_Raw_Data'
all_files = glob.glob(os.path.join(path, "*.csv"))

# Read each CSV and append it to a list
df_list = [pd.read_csv(file) for file in all_files]

# Concatenate all dataframes in the list into one massive dataframe
cyclistic_df = pd.concat(df_list, ignore_index=True)

# Check the total number of rows and columns
print(f"Total rows and columns: {cyclistic_df.shape}")

Total rows and columns: (5780912, 13)


In [3]:
# 1. Convert 'started_at' and 'ended_at' columns to datetime format
cyclistic_df['started_at'] = pd.to_datetime(cyclistic_df['started_at'])
cyclistic_df['ended_at'] = pd.to_datetime(cyclistic_df['ended_at'])

# 2. Create 'ride_length' column (duration in minutes for easier analysis)
cyclistic_df['ride_length'] = (cyclistic_df['ended_at'] - cyclistic_df['started_at']).dt.total_seconds() / 60

# 3. Create 'day_of_week' column (e.g., Monday, Tuesday...)
cyclistic_df['day_of_week'] = cyclistic_df['started_at'].dt.day_name()

# 4. Create a 'month' column to look at seasonality later
cyclistic_df['month'] = cyclistic_df['started_at'].dt.month_name()

# Look at the first 5 rows to ensure it worked
cyclistic_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week,month
0,16CBE9844D401954,electric_bike,2025-03-18 08:39:20.065,2025-03-18 08:51:37.633,NaN,NaN,Canal St & Jackson Blvd,13138,41.91,-87.67,41.878125,-87.639968,member,12.292800,Tuesday,March
1,1CB408029E2B5F74,electric_bike,2025-03-24 16:04:22.239,2025-03-24 16:27:41.347,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member,23.318467,Monday,March
2,7B6A76CD0F204D08,electric_bike,2025-03-10 16:06:19.708,2025-03-10 16:29:17.457,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.86,-87.68,41.914027,-87.705126,member,22.962483,Monday,March
3,4F7084E3D75CDE31,electric_bike,2025-03-21 14:28:14.579,2025-03-21 14:35:06.160,NaN,NaN,Canal St & Jackson Blvd,13138,41.87,-87.63,41.878125,-87.639968,member,6.859683,Friday,March
4,E419A570A5A0475B,electric_bike,2025-03-14 17:54:14.484,2025-03-14 18:17:53.254,NaN,NaN,Albany Ave & Bloomingdale Ave,15655,41.89,-87.67,41.914027,-87.705126,casual,23.646167,Friday,March


In [7]:
# 1. Check for duplicates
cyclistic_df = cyclistic_df.drop_duplicates()

# 2. Check for missing values (Nulls)
# Note: Sometimes station names are missing. For this analysis, it's safer to drop rows missing crucial data to ensure our numbers are highly accurate.
cyclistic_df = cyclistic_df.dropna()

# 3. Remove "bad" ride lengths
# Some rides have negative time (end time is before start time) or last less than 1 minute (false starts/locked bikes). 
# We also remove rides longer than 24 hours (1440 minutes) as they are likely stolen/unreturned bikes.
cleaned_df = cyclistic_df[(cyclistic_df['ride_length'] >= 1) & (cyclistic_df['ride_length'] <= 1440)]

# Let's see how many rows we have left after cleaning
print(f"Total rows after cleaning: {cleaned_df.shape}")

Total rows after cleaning: (3817240, 16)


In [8]:
# Save to CSV
cleaned_df.to_csv('Data/02_Prepared_Data/cleaned_cyclistic_data.csv', index=False)
print("Data successfully cleaned and saved!")

Data successfully cleaned and saved!
